<a href="https://colab.research.google.com/github/rhodes-byu/cs-stat-180/blob/main/notebooks/18-ai-nba-stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basketball NBA Stats Chatbot
### Generative AI Lab

**Pattern:** question -> LLM writes pandas code -> we run it -> LLM explains results

**Dataset:** [NBA Players Stats Since 1950](https://www.kaggle.com/datasets/drgilermo/nba-players-stats) - upload `Seasons_Stats.csv` when prompted.

**API key:** [console.anthropic.com](https://console.anthropic.com)


In [ ]:
!pip install anthropic --quiet
print("Done")


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("drgilermo/nba-players-stats")

print("Path to dataset files:", path)

In [ ]:
from google.colab import files
import pandas as pd, numpy as np, io

print("Upload Seasons_Stats.csv when the picker opens...")
uploaded = files.upload()

df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]), low_memory=False)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)
df["Player"] = df["Player"].astype(str).str.strip().str.replace("*", "", regex=False)
stat_cols = ["G","GS","MP","PTS","TRB","ORB","DRB","AST","STL","BLK","TOV","PF",
             "FG","FGA","FG%","3P","3PA","3P%","2P","2PA","2P%","FT","FTA","FT%",
             "eFG%","TS%","PER","USG%","WS","WS/48","OWS","DWS","BPM","VORP","Age"]
for col in stat_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows - {df['Player'].nunique():,} players, {df['Year'].min()}-{df['Year'].max()}")
df.head(3)


In [ ]:
import getpass
API_KEY = getpass.getpass("Paste your Anthropic API key: ")


## The Data Dictionary

Instead of hardcoded keyword rules, we give the LLM a plain-English description
of every column so it can write the right pandas query for any question.

Getting this right matters: the original version said BLK was 'blocks per game'
when it's actually a season total — so the LLM reported Mark Eaton averaging
278 blocks per game. The data dictionary is the LLM's only view of the schema.


In [ ]:
# Data dictionary injected into every prompt.
# Sourced from the official Basketball Reference glossary.

lines = [
    "DataFrame: df | ~24,000 rows | one row per player per season",
    "Year: end year of season (2000 = the 1999-2000 season)",
    "Player: full name string e.g. Michael Jordan",
    "Pos: position - PG, SG, SF, PF, C (sometimes combined e.g. SF-PF)",
    "Age: player age on January 31 of the given season",
    "Tm: team abbreviation e.g. CHI, LAL, BOS. TOT = season total row for mid-season trades",
    "",
    "G: games played that season",
    "GS: games started that season (available since 1982 only)",
    "MP: total minutes played that season (NOT per game)",
    "",
    "=== COUNTING STATS: ALL ARE SEASON TOTALS, NOT PER GAME ===",
    "To get per-game rate you MUST divide by G. Example: df['BLK']/df['G'] = blocks per game",
    "",
    "PTS: total points scored that season",
    "TRB: total rebounds that season (available since 1950-51)",
    "ORB: offensive rebounds that season (available since 1973-74 only - NaN before that)",
    "DRB: defensive rebounds that season (available since 1973-74 only - NaN before that)",
    "AST: total assists that season",
    "STL: total steals that season (available since 1973-74 only - NaN before that)",
    "BLK: total blocks that season (available since 1973-74 only - NaN before that)",
    "TOV: total turnovers that season (available since 1977-78 only - NaN before that)",
    "PF: personal fouls that season",
    "FG: field goals made (both 2-point and 3-point)",
    "FGA: field goal attempts",
    "3P: three-point field goals made (available since 1979-80 only - NaN before that)",
    "3PA: three-point field goal attempts (available since 1979-80 only)",
    "2P: two-point field goals made",
    "2PA: two-point field goal attempts",
    "FT: free throws made",
    "FTA: free throw attempts",
    "",
    "=== RATE/EFFICIENCY STATS: ALREADY NORMALIZED, DO NOT DIVIDE BY G ===",
    "",
    "FG%: field goal percentage = FG/FGA",
    "3P%: three-point percentage = 3P/3PA (column name is '3P%', available since 1979-80)",
    "2P%: two-point field goal percentage = 2P/2PA",
    "FT%: free throw percentage = FT/FTA",
    "eFG%: effective field goal pct = (FG + 0.5*3P)/FGA (adjusts for 3s being worth more)",
    "TS%: true shooting pct = PTS/(2*TSA) where TSA=FGA+0.44*FTA (best overall shooting efficiency)",
    "PER: Player Efficiency Rating, per-minute rating (league average = 15, higher is better)",
    "USG%: usage percentage, estimate of % of team plays used while on floor (avg ~20%)",
    "",
    "=== ACCUMULATION STATS: SEASON VALUES, DO NOT DIVIDE BY G ===",
    "",
    "WS: Win Shares, estimate of wins contributed that season (already a season total)",
    "WS/48: Win Shares per 48 minutes (league average ~0.100, already a rate)",
    "OWS: Offensive Win Shares that season",
    "DWS: Defensive Win Shares that season",
    "BPM: Box Plus/Minus, points per 100 possessions above league average (already a rate)",
    "VORP: Value Over Replacement Player, prorated to 82-game season (already normalized, do not divide by G)",
    "",
    "=== QUERY PATTERNS ===",
    "",
    "Per-game career leaders (add column, then groupby):",
    "  tmp=df.copy(); tmp['BPG']=tmp['BLK']/tmp['G']",
    "  result=tmp.groupby('Player')['BPG'].mean().sort_values(ascending=False).head(20)",
    "",
    "Season leaderboard (exclude TOT rows to avoid double-counting traded players):",
    "  season=df[(df['Year']==YYYY)&(df['Tm']!='TOT')]",
    "  result=season.nlargest(20, 'PTS')[['Player','Tm','G','PTS']]",
    "",
    "Player career lookup (use str.contains for fuzzy match):",
    "  result=df[df['Player'].str.contains('Jordan',case=False,na=False)].sort_values('Year')",
    "",
    "Multi-player comparison:",
    "  result=df[df['Player'].str.contains('Jordan|Bryant',case=False,na=False)]",
    "",
    "Dataset covers 1950-2017. Post-2017 players are not present.",
    "Many advanced stats (BLK, STL, ORB, DRB, TOV, 3P) are NaN before their availability year.",
]

DATA_DICTIONARY = "\n".join(lines)
print(f"Data dictionary ready ({len(DATA_DICTIONARY.split())} words)")


## The `ask()` function

Two LLM calls:
- **Call 1** `temperature=0` - writes pandas code to extract the right data
- **Call 2** `temperature=0.2` - explains the results in plain English

Use `debug=True` to see what code the model generated.


In [ ]:
import anthropic, traceback

client = anthropic.Anthropic(api_key=API_KEY)


def ask(question, debug=False):
    # -- Call 1: generate pandas query (temperature=0 = deterministic code) ------
    query_prompt = (
        "You have a pandas DataFrame called `df`.\n\n"
        + DATA_DICTIONARY + "\n\n"
        + "Write Python code to extract the data needed to answer:\n"
        + f'  "{question}"\n\n'
        + "Rules:\n"
        + "- Assign final result to a variable called `result`\n"
        + "- Return ONLY executable Python, no markdown fences, no comments\n"
        + "- 1-6 lines max\n"
        + "- NEVER divide rate/efficiency stats (WS, VORP, PER, BPM, FG%, TS%, etc.) by G\n"
        + "- ALWAYS divide counting stats (PTS, BLK, TRB, AST etc.) by G for per-game rates\n"
        + "- For per-game leaderboards: add a per-game column first, then groupby and mean()\n"
        + "- For season leaderboards: exclude TOT rows with df[df['Tm'] != 'TOT']\n"
        + "- For player lookup: use str.contains(case=False, na=False)\n"
        + "- For multi-player: use pipe in str.contains e.g. 'Jordan|Bryant'\n"
    )

    r1 = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        temperature=0,
        messages=[{"role": "user", "content": query_prompt}]
    )

    generated_code = r1.content[0].text.strip()
    if generated_code.startswith('```'):
        generated_code = '\n'.join(generated_code.split('\n')[1:])
    if generated_code.endswith('```'):
        generated_code = '\n'.join(generated_code.split('\n')[:-1])
    generated_code = generated_code.strip()

    if debug:
        print('-- Generated code ---------------------------------')
        print(generated_code)
        print('---------------------------------------------------\n')

    # -- Execute the generated code ------------------------------------------
    local_vars = {"df": df, "np": np, "pd": pd}
    try:
        exec(generated_code, {}, local_vars)
        result = local_vars.get("result", None)
        if result is None:
            data_context = "Query produced no result."
        elif isinstance(result, pd.DataFrame):
            data_context = f"Result ({len(result)} rows):\n{result.head(30).to_string(index=False)}"
        elif isinstance(result, pd.Series):
            data_context = f"Result:\n{result.head(30).to_string()}"
        else:
            data_context = f"Result: {result}"
    except Exception as e:
        if debug:
            traceback.print_exc()
        data_context = (
            f"Code failed ({e}). Dataset sample:\n{df.sample(15).to_string(index=False)}"
        )

    # -- Call 2: generate plain-English answer --------------------------------
    r2 = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        temperature=0.2,
        system="You are an NBA analyst. Answer using ONLY the provided data. Do not invent statistics.",
        messages=[{"role": "user", "content": (
            f"Question: {question}\n\nData:\n{data_context}\n\n"
            "Answer conversationally, cite specific numbers, 2-4 paragraphs."
        )}]
    )

    print(r2.content[0].text)


print("Ready. Call ask('question') - add debug=True to see the generated code.")


## Ask questions

In [ ]:
ask("How did Michael Jordan perform over his career?")

In [ ]:
ask("Who were the top scorers in the 1988 season?")

In [ ]:
ask("Compare LeBron James and Kobe Bryant")

In [ ]:
ask("Which centers had the highest career blocks per game?", debug=True)

In [ ]:
ask("Who had the best true shooting percentage with at least 5 seasons?")

In [ ]:
ask("YOUR QUESTION HERE")

## Discussion

1. Run the blocks question with `debug=True`. Does it divide by G? What changed vs before?
2. Try `ask('Who had the highest VORP?')` - does it (correctly) NOT divide by G?
3. Try removing the availability notes (e.g. 'available since 1973-74') from DATA_DICTIONARY
   then ask about blocks for players from the 1960s. What happens?
4. Why `temperature=0` for code generation but `temperature=0.2` for the answer?
5. We run LLM-generated code with `exec()`. Why is this dangerous in production?
